In [1]:
import math
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

try:
    import statsmodels.api as sm
except ImportError:
    sm = None

workspace_root = "/models/hiera-luna25"
results_dir = f"{workspace_root}/results/finetune-hiera-3D-20260814"
out_dir = Path(f"{workspace_root}/figures/diagnostics")
FOV_Z = 12.5
MIN_N = 10          # min nodules per group in a cell
rng = np.random.default_rng(0)

df_pred = pd.read_csv(f"{results_dir}/all_folds_predictions.csv")
csv_path = out_dir / "truncation_metrics.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"{csv_path} missing: run `cd jobs && sbatch run_analysis.sh analysis/diagnostics_truncation.py` first")
tm = pd.read_csv(csv_path)
m = tm.merge(df_pred[["AnnotationID", "probability"]], on="AnnotationID", how="left")
m = m.dropna(subset=["absolute_error", "nodule_volume_mm3", "frac_outside", "z_extent_mm"])
m = m[m["nodule_volume_mm3"] > 0].copy()
m["trunc"] = m["trunc_z"].astype(bool)
m["elong"] = m["z_extent_mm"] > FOV_Z
m["log_vol"] = np.log(m["nodule_volume_mm3"])
m["diameter_mm"] = 2 * (3 * m["nodule_volume_mm3"] / (4 * math.pi)) ** (1 / 3)
m["abs_off_z"] = m["centroid_offset_z_mm"].abs()
print(tm.shape, m.shape, "statsmodels:", sm is not None)

def boot_ci(x, stat=np.mean, n=2000):
    x = np.asarray(x)
    if len(x) < 2:
        return (np.nan, np.nan)
    idx = rng.integers(0, len(x), (n, len(x)))
    b = np.array([stat(x[i]) for i in idx]) if stat is not np.mean else x[idx].mean(1)
    return tuple(np.percentile(b, [2.5, 97.5]))

def boot_diff(a, b, n=2000):
    a, b = np.asarray(a), np.asarray(b)
    da = a[rng.integers(0, len(a), (n, len(a)))].mean(1)
    db = b[rng.integers(0, len(b), (n, len(b)))].mean(1)
    return tuple(np.percentile(da - db, [2.5, 97.5]))

(4770, 18) (4770, 24) statsmodels: False


In [2]:
# (a) truncation prevalence
print(f"truncated (trunc_z): {m.trunc.mean():.3f}  (n={m.trunc.sum()}/{len(m)})")
print(f"elongated (z_extent > {FOV_Z}): {m.elong.mean():.3f}   touches_border: {m['touches_border'].astype(bool).mean():.3f}   frac_outside>0: {(m.frac_outside > 0).mean():.3f}")
display(m.groupby("label").agg(n=("trunc", "size"), frac_trunc=("trunc", "mean"), frac_elong=("elong", "mean")).round(3))
m["dbin"] = pd.cut(m["diameter_mm"], [0, 6, 8, 12.5, 15, 20, np.inf])
display(m.groupby("dbin", observed=False).agg(n=("trunc", "size"), frac_trunc=("trunc", "mean"), frac_elong=("elong", "mean"),
                                              med_z_extent=("z_extent_mm", "median")).round(3))
# check of the ">=12.5 mm diameter" claim against z extent
big = m["diameter_mm"] >= FOV_Z
print("crosstab eq.diameter>=12.5 vs trunc_z"); display(pd.crosstab(big, m.trunc, margins=True))
print(f"P(trunc | diam>=12.5) = {m.loc[big, 'trunc'].mean():.3f};  P(diam>=12.5 | trunc) = {big[m.trunc].mean():.3f}")
print(f"median z_extent / diameter: {(m.z_extent_mm / m.diameter_mm).median():.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(m["z_extent_mm"], bins=60, color="C0", alpha=.8)
ax[0].axvline(FOV_Z, color="r", ls="--", label=f"FOV z = {FOV_Z} mm"); ax[0].set_xlabel("z extent (mm)"); ax[0].set_ylabel("nodules"); ax[0].legend()
g = m.groupby("dbin", observed=False)[["trunc", "elong"]].mean()
g.plot.bar(ax=ax[1]); ax[1].set_xlabel("equivalent diameter bin (mm)"); ax[1].set_ylabel("fraction"); ax[1].set_title("truncation prevalence")
fig.tight_layout(); fig.savefig(out_dir / "truncation_prevalence.png", dpi=150)

truncated (trunc_z): 0.164  (n=784/4770)
elongated (z_extent > 12.5): 0.149   touches_border: 0.206   frac_outside>0: 0.157


,n,frac_trunc,frac_elong
label,,,
0,4324,0.128,0.114
1,446,0.520,0.491


,n,frac_trunc,frac_elong,med_z_extent
dbin,,,,
"(0.0, 6.0]",2256,0.000,0.000,5.0
"(6.0, 8.0]",1053,0.018,0.009,7.2
"(8.0, 12.5]",820,0.233,0.182,10.0
"(12.5, 15.0]",203,0.719,0.650,14.0
"(15.0, 20.0]",214,0.958,0.925,18.0
"(20.0, inf]",224,0.991,0.991,32.0


crosstab eq.diameter>=12.5 vs trunc_z


trunc,False,True,All
diameter_mm,,,
False,3918,211,4129
True,68,573,641
All,3986,784,4770


P(trunc | diam>=12.5) = 0.894;  P(diam>=12.5 | trunc) = 0.731


median z_extent / diameter: 1.009


In [3]:
# (b) main test: equal-count volume bins, truncated vs not
NB = 10
m["vbin"] = pd.qcut(m["nodule_volume_mm3"], NB, labels=False, duplicates="drop")
rows = []
for b, g in m.groupby("vbin"):
    t, n = g[g.trunc], g[~g.trunc]
    r = {"vbin": b, "vol_lo": g.nodule_volume_mm3.min(), "vol_hi": g.nodule_volume_mm3.max(), "n_trunc": len(t), "n_non": len(n)}
    if len(t) >= MIN_N and len(n) >= MIN_N:
        lo, hi = boot_diff(t.absolute_error, n.absolute_error)
        r.update(mean_trunc=t.absolute_error.mean(), mean_non=n.absolute_error.mean(), med_trunc=t.absolute_error.median(),
                 med_non=n.absolute_error.median(), diff=t.absolute_error.mean() - n.absolute_error.mean(), diff_lo=lo, diff_hi=hi)
    rows.append(r)
tb = pd.DataFrame(rows).reindex(columns=["vbin","vol_lo","vol_hi","n_trunc","n_non","mean_trunc","mean_non","med_trunc","med_non","diff","diff_lo","diff_hi"]); display(tb.round(3))

ok = tb.dropna(subset=["diff"])
if len(ok):
    sub = m[m.vbin.isin(ok.vbin)]
    w = ok[["n_trunc", "n_non"]].min(axis=1).values
    pooled = np.average(ok["diff"], weights=w)
    # stratified bootstrap of the weighted within-bin difference
    bs = []
    for _ in range(1000):
        d_ = []
        for b in ok.vbin:
            g = sub[sub.vbin == b]
            t = g[g.trunc].absolute_error.values; n = g[~g.trunc].absolute_error.values
            d_.append(rng.choice(t, len(t)).mean() - rng.choice(n, len(n)).mean())
        bs.append(np.average(d_, weights=w))
    print(f"valid bins: {len(ok)}; pooled within-bin diff (trunc - non) = {pooled:.4f}  95% CI [{np.percentile(bs, 2.5):.4f}, {np.percentile(bs, 97.5):.4f}]")
else:
    print("no bin has enough nodules in both groups")
crude = m[m.trunc].absolute_error.mean() - m[~m.trunc].absolute_error.mean()
print(f"crude (unadjusted) diff = {crude:.4f}")

m["vpct"] = m["nodule_volume_mm3"].rank(pct=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for flag, c, lab in [(False, "C0", "not truncated"), (True, "C3", "truncated")]:
    s = m[m.trunc == flag]
    ax[0].scatter(s.vpct, s.absolute_error, s=6, alpha=.35, color=c, label=lab)
ax[0].set_xlabel("volume percentile"); ax[0].set_ylabel("absolute error"); ax[0].legend()
x = np.arange(len(tb))
for flag, c, off in [(False, "C0", -.1), (True, "C3", .1)]:
    k = "non" if not flag else "trunc"
    mu, lo, hi = [], [], []
    for b, g in m.groupby("vbin"):
        s = g[g.trunc == flag].absolute_error
        if len(s) >= MIN_N:
            l, h = boot_ci(s); mu.append(s.mean()); lo.append(l); hi.append(h)
        else:
            mu.append(np.nan); lo.append(np.nan); hi.append(np.nan)
    mu, lo, hi = map(np.array, (mu, lo, hi))
    ax[1].errorbar(x + off, mu, yerr=[mu - lo, hi - mu], fmt="o-", color=c, capsize=3, label=("truncated" if flag else "not truncated"))
ax[1].set_xlabel("volume decile (cells with n>=%d)" % MIN_N); ax[1].set_ylabel("mean absolute error"); ax[1].legend()
fig.tight_layout(); fig.savefig(out_dir / "truncation_error_by_volume.png", dpi=150)

,vbin,vol_lo,vol_hi,n_trunc,n_non,mean_trunc,mean_non,med_trunc,med_non,diff,diff_lo,diff_hi
0,0,3.959,43.365,0,477,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,43.370,59.328,0,477,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,59.368,73.327,0,477,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,73.438,94.055,0,477,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,94.123,121.158,1,476,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,121.171,171.562,4,473,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,171.641,276.389,18,459,0.258,0.356,0.192,0.295,-0.098,-0.172,-0.025
7,7,276.777,570.296,76,401,0.410,0.417,0.377,0.406,-0.007,-0.067,0.055
8,8,574.238,1575.195,223,254,0.428,0.445,0.366,0.457,-0.017,-0.061,0.027
9,9,1581.622,241810.126,462,15,0.417,0.449,0.333,0.461,-0.032,-0.168,0.100


valid bins: 4; pooled within-bin diff (trunc - non) = -0.0200  95% CI [-0.0547, 0.0114]
crude (unadjusted) diff = 0.1432


/usr/local/lib/python3.9/site-packages/numpy/core/_methods.py:44: RuntimeWarning: invalid value encountered in reduce
  return umr_minimum(a, axis, None, out, keepdims, initial, where)
/usr/local/lib/python3.9/site-packages/numpy/core/_methods.py:40: RuntimeWarning: invalid value encountered in reduce
  return umr_maximum(a, axis, None, out, keepdims, initial, where)


In [4]:
# (c) adjusted effect
from scipy.stats import rankdata
def resid(y, X):
    X = np.column_stack([np.ones(len(X)), X]); beta = np.linalg.lstsq(X, y, rcond=None)[0]; return y - X @ beta
rk = lambda v: rankdata(v)
rho, p = spearmanr(m.absolute_error, m.frac_outside); print(f"Spearman(error, frac_outside) = {rho:.3f} (p={p:.2g})")
rho_v, _ = spearmanr(m.absolute_error, m.log_vol); print(f"Spearman(error, log_vol) = {rho_v:.3f}")
rp, pp = spearmanr(resid(rk(m.absolute_error), rk(m.log_vol)), resid(rk(m.frac_outside), rk(m.log_vol)))
print(f"partial Spearman(error, frac_outside | log_vol) = {rp:.3f} (p={pp:.2g})")

def ols(d, cols, y="absolute_error", nboot=500):
    X = np.column_stack([np.ones(len(d))] + [d[c].astype(float).values for c in cols]); yy = d[y].values
    beta = np.linalg.lstsq(X, yy, rcond=None)[0]
    if sm is not None:
        r = sm.OLS(yy, X).fit(cov_type="HC3"); ci = r.conf_int(); se = r.bse; pv = r.pvalues
        lo, hi = ci[:, 0], ci[:, 1]
    else:
        bs = []
        for _ in range(nboot):
            i = rng.integers(0, len(d), len(d)); bs.append(np.linalg.lstsq(X[i], yy[i], rcond=None)[0])
        bs = np.array(bs); lo, hi = np.percentile(bs, 2.5, 0), np.percentile(bs, 97.5, 0); se = bs.std(0); pv = np.full(len(beta), np.nan)
    return pd.DataFrame({"coef": beta, "se": se, "lo": lo, "hi": hi, "p": pv}, index=["const"] + cols)

cols = ["log_vol", "frac_outside", "trunc_i", "abs_off_z"]
m["trunc_i"] = m.trunc.astype(int)
print("pooled (+label)"); display(ols(m, cols + ["label"]).round(4))
for l in (0, 1):
    print(f"label {l} (n={int((m.label == l).sum())})"); display(ols(m[m.label == l], cols).round(4))
print("volume only vs +truncation terms, R2:")
for cs in (["log_vol"], cols):
    X = np.column_stack([np.ones(len(m))] + [m[c].astype(float) for c in cs]); r = m.absolute_error - X @ np.linalg.lstsq(X, m.absolute_error, rcond=None)[0]
    print(cs, round(1 - r.var() / m.absolute_error.var(), 4))

Spearman(error, frac_outside) = 0.226 (p=2.5e-56)
Spearman(error, log_vol) = 0.467
partial Spearman(error, frac_outside | log_vol) = -0.085 (p=4.8e-09)
pooled (+label)


,coef,se,lo,hi,p
const,-0.1511,0.0146,-0.1820,-0.1270,NaN
log_vol,0.0897,0.0032,0.0840,0.0964,NaN
frac_outside,-0.4035,0.0599,-0.5196,-0.2876,NaN
trunc_i,-0.0117,0.0163,-0.0437,0.0179,NaN
abs_off_z,0.0091,0.0037,0.0018,0.0170,NaN
label,-0.0927,0.0134,-0.1197,-0.0671,NaN


label 0 (n=4324)


,coef,se,lo,hi,p
const,-0.2050,0.0156,-0.2353,-0.1757,NaN
log_vol,0.1005,0.0035,0.0936,0.1075,NaN
frac_outside,-0.1652,0.0642,-0.2960,-0.0479,NaN
trunc_i,-0.0226,0.0181,-0.0580,0.0130,NaN
abs_off_z,-0.0016,0.0040,-0.0089,0.0066,NaN


label 1 (n=446)


,coef,se,lo,hi,p
const,0.8313,0.0569,0.7209,0.9452,NaN
log_vol,-0.0802,0.0097,-0.0993,-0.0612,NaN
frac_outside,0.2645,0.0694,0.1263,0.3951,NaN
trunc_i,-0.0022,0.0233,-0.0474,0.0453,NaN
abs_off_z,-0.0020,0.0035,-0.0078,0.0067,NaN


volume only vs +truncation terms, R2:
['log_vol'] 0.1647
['log_vol', 'frac_outside', 'trunc_i', 'abs_off_z'] 0.1893


In [5]:
# (d) dose-response within a fixed volume band
t_vol, n_vol = m.loc[m.trunc, "nodule_volume_mm3"], m.loc[~m.trunc, "nodule_volume_mm3"]
lo_v, hi_v = t_vol.quantile(.05), n_vol.quantile(.95)   # overlap region of the two volume distributions
band = m[(m.nodule_volume_mm3 >= lo_v) & (m.nodule_volume_mm3 <= hi_v)].copy()
print(f"volume band [{lo_v:.0f}, {hi_v:.0f}] mm3: n={len(band)} (trunc={int(band.trunc.sum())})")
band["fbin"] = pd.cut(band.frac_outside, [-1e-9, 0, .1, .25, .5, 1.0], labels=["0", "(0,.1]", "(.1,.25]", "(.25,.5]", ">.5"])
rows = []
for k, g in band.groupby("fbin", observed=False):
    lo, hi = boot_ci(g.absolute_error) if len(g) >= MIN_N else (np.nan, np.nan)
    rows.append({"frac_outside": k, "n": len(g), "mean_err": g.absolute_error.mean() if len(g) >= MIN_N else np.nan, "lo": lo, "hi": hi,
                 "mean_log_vol": g.log_vol.mean()})
db = pd.DataFrame(rows); display(db.round(3))
if len(band) > 20:
    print("within-band Spearman(error, frac_outside) = %.3f (p=%.2g)" % spearmanr(band.absolute_error, band.frac_outside))
fig, ax = plt.subplots(figsize=(6, 4))
d_ = db.dropna(subset=["mean_err"])
ax.errorbar(range(len(d_)), d_.mean_err, yerr=[d_.mean_err - d_.lo, d_.hi - d_.mean_err], fmt="o-", capsize=3)
ax.set_xticks(range(len(d_))); ax.set_xticklabels([f"{a}\n(n={n})" for a, n in zip(d_.frac_outside, d_.n)])
ax.set_xlabel("frac_outside (volume-band restricted)"); ax.set_ylabel("mean absolute error"); fig.tight_layout()
fig.savefig(out_dir / "truncation_dose_response.png", dpi=150)

volume band [340, 661] mm3: n=406 (trunc=85)


,frac_outside,n,mean_err,lo,hi,mean_log_vol
0,0,326,0.426,0.401,0.451,6.150
1,"(0,.1]",35,0.460,0.382,0.542,6.224
2,"(.1,.25]",32,0.391,0.307,0.475,6.221
3,"(.25,.5]",12,0.433,0.308,0.551,6.207
4,>.5,1,NaN,NaN,NaN,6.325


within-band Spearman(error, frac_outside) = -0.016 (p=0.74)


In [6]:
# (e) elongation vs off-centre truncation
m_all = m
m["grp"] = np.select([m.elong, m.trunc], ["elongated (z_extent>12.5)", "off-centre/xy only"], "not truncated")
m = m[(m.nodule_volume_mm3 >= lo_v) & (m.nodule_volume_mm3 <= hi_v)].copy()   # restrict to volume-overlap band (no extrapolation)
print(f"restricted to overlap band [{lo_v:.0f}, {hi_v:.0f}] mm3: n={len(m)}")
# volume-adjusted residual: fit error ~ log_vol on non-truncated, apply to all
nt = m[m.grp == "not truncated"]
b1 = np.polyfit(nt.log_vol, nt.absolute_error, 1); m["err_resid"] = m.absolute_error - np.polyval(b1, m.log_vol)
rows = []
for k, g in m.groupby("grp"):
    l1, h1 = boot_ci(g.absolute_error); l2, h2 = boot_ci(g.err_resid)
    rows.append({"group": k, "n": len(g), "median_log_vol": g.log_vol.median(), "mean_err": g.absolute_error.mean(), "lo": l1, "hi": h1,
                 "mean_resid_vs_nontrunc": g.err_resid.mean(), "res_lo": l2, "res_hi": h2, "mean_abs_off_z": g.abs_off_z.mean()})
ge = pd.DataFrame(rows); display(ge.round(3))
print("(residual = error minus linear-in-log-volume fit on non-truncated nodules; extrapolates for large volumes)")
# regression with both flags
m["off_only"] = (m.grp == "off-centre/xy only").astype(int); m["elong_i"] = m.elong.astype(int)
display(ols(m, ["log_vol", "elong_i", "off_only", "label"]).round(4))
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.errorbar(range(len(ge)), ge.mean_resid_vs_nontrunc, yerr=[ge.mean_resid_vs_nontrunc - ge.res_lo, ge.res_hi - ge.mean_resid_vs_nontrunc], fmt="o", capsize=4)
ax.axhline(0, color="gray", lw=.8); ax.set_xticks(range(len(ge))); ax.set_xticklabels([f"{g}\n(n={n})" for g, n in zip(ge.group, ge.n)], fontsize=8)
ax.set_ylabel("volume-adjusted error residual"); fig.tight_layout(); fig.savefig(out_dir / "truncation_groups.png", dpi=150)

restricted to overlap band [340, 661] mm3: n=406


,group,n,median_log_vol,mean_err,lo,hi,mean_resid_vs_nontrunc,res_lo,res_hi,mean_abs_off_z
0,elongated (z_extent>12.5),65,6.249,0.428,0.370,0.480,-0.005,-0.061,0.049,1.19
1,not truncated,321,6.138,0.426,0.400,0.451,-0.000,-0.024,0.025,0.45
2,off-centre/xy only,20,6.233,0.421,0.308,0.535,-0.011,-0.127,0.110,1.77


(residual = error minus linear-in-log-volume fit on non-truncated nodules; extrapolates for large volumes)


,coef,se,lo,hi,p
const,-0.0617,0.3782,-0.8127,0.6902,NaN
log_vol,0.0819,0.0615,-0.0409,0.2055,NaN
elong_i,-0.0047,0.0308,-0.0638,0.0552,NaN
off_only,-0.0115,0.0642,-0.1321,0.1191,NaN
label,-0.0957,0.0233,-0.1419,-0.0491,NaN


In [7]:
# (f) AUC per truncation group (overall and within volume-matched bins)
m = m_all
def auc(g): return roc_auc_score(g.label, g.probability) if g.label.nunique() == 2 and len(g) >= 20 else np.nan
def auc_ci(g, n=500):
    g = g.reset_index(drop=True); b = []
    for _ in range(n):
        s = g.iloc[rng.integers(0, len(g), len(g))]
        if s.label.nunique() == 2: b.append(roc_auc_score(s.label, s.probability))
    return np.percentile(b, [2.5, 97.5]) if b else (np.nan, np.nan)
mm = m.dropna(subset=["probability"])
rows = []
for name, sel in [("all", mm), ("non-trunc", mm[~mm.trunc]), ("trunc", mm[mm.trunc]),
                  ("vol-matched non-trunc", mm[(~mm.trunc) & mm.vbin.isin(ok.vbin)]), ("vol-matched trunc", mm[mm.trunc & mm.vbin.isin(ok.vbin)])]:
    lo, hi = auc_ci(sel) if len(sel) >= 20 else (np.nan, np.nan)
    rows.append({"group": name, "n": len(sel), "n_pos": int(sel.label.sum()), "auc": auc(sel), "lo": lo, "hi": hi})
display(pd.DataFrame(rows).round(3))

,group,n,n_pos,auc,lo,hi
0,all,4770,446,0.895,0.880,0.907
1,non-trunc,3986,214,0.879,0.857,0.899
2,trunc,784,232,0.815,0.781,0.847
3,vol-matched non-trunc,1129,138,0.802,0.761,0.836
4,vol-matched trunc,779,232,0.814,0.786,0.845
